# OpenPose Video Annotation and Search
## Using YOLO Pose for Easy Setup

This notebook demonstrates:
1. Video annotation with pose estimation
2. Keypoint extraction and storage
3. Video search based on pose similarity
4. Action matching and analysis

## 1. Setup and Installation

In [ ]:
# Install required packages (run once)
!pip install ultralytics opencv-python dtaidistance numpy scipy matplotlib

In [ ]:
# Imports
from ultralytics import YOLO
import cv2
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Video, display

# Import our pose search module
from pose_search import PoseSearchEngine

print("All imports successful!")

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Pose Estimation Model

In [ ]:
# Load YOLO Pose model
# Options: yolov8n-pose.pt (nano, fastest)
#          yolov8s-pose.pt (small)
#          yolov8m-pose.pt (medium)
#          yolov8l-pose.pt (large)
#          yolov8x-pose.pt (extra large, most accurate)

model = YOLO('yolov8x-pose.pt')  # Downloads automatically on first run
print("Model loaded successfully!")

## 3. Annotate Videos with Pose Detection

In [ ]:
# Setup paths
input_videos = [
    'videos/sample1.mp4',
    'videos/sample2.mp4',
    'videos/sample3.mp4'
]

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

In [ ]:
# Annotate a single video (quick test)
test_video = 'videos/sample1.mp4'

results = model.predict(
    source=test_video,
    save=True,
    show=False,
    conf=0.5,
    project=str(output_dir),
    name='test_run',
    exist_ok=True
)

print(f"Processed {len(results)} frames")
print(f"Output saved to: {output_dir / 'test_run'}")

In [ ]:
# Display annotated video
annotated_video = list((output_dir / 'test_run').glob('*.avi'))[0]
display(Video(str(annotated_video), width=640))

## 4. Extract and Save Keypoints

In [ ]:
def extract_keypoints_from_video(video_path, output_json_path, model):
    """
    Extract keypoints from video and save to JSON
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    all_frames = []
    frame_idx = 0
    
    print(f"Processing {video_path}...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run inference
        results = model(frame, verbose=False)
        
        frame_data = {
            'frame': frame_idx,
            'timestamp': frame_idx / fps,
            'people': []
        }
        
        # Extract keypoints for each person
        if results[0].keypoints is not None:
            keypoints = results[0].keypoints.xy.cpu().numpy()  # (num_people, 17, 2)
            confidences = results[0].keypoints.conf.cpu().numpy()
            
            for person_idx in range(len(keypoints)):
                person_data = {
                    'person_id': person_idx,
                    'pose_keypoints_2d': keypoints[person_idx].flatten().tolist(),
                    'confidence': confidences[person_idx].tolist()
                }
                frame_data['people'].append(person_data)
        
        all_frames.append(frame_data)
        frame_idx += 1
        
        if frame_idx % 30 == 0:
            print(f"  Processed {frame_idx} frames...", end='\r')
    
    cap.release()
    
    # Save to JSON
    with open(output_json_path, 'w') as f:
        json.dump(all_frames, f, indent=2)
    
    print(f"\nSaved {len(all_frames)} frames to {output_json_path}")
    return all_frames

In [ ]:
# Extract keypoints from all videos
keypoints_dir = output_dir / 'keypoints'
keypoints_dir.mkdir(exist_ok=True)

for video_path in input_videos:
    video_name = Path(video_path).stem
    json_path = keypoints_dir / f"{video_name}_keypoints.json"
    
    keypoints = extract_keypoints_from_video(video_path, json_path, model)
    print(f"Completed {video_name}\n")

## 5. Visualize Keypoints

In [ ]:
# Load and visualize keypoints for a single frame
with open(keypoints_dir / 'sample1_keypoints.json', 'r') as f:
    sample_data = json.load(f)

# Get first frame with detected person
frame_with_person = None
for frame in sample_data:
    if frame['people']:
        frame_with_person = frame
        break

if frame_with_person:
    keypoints = np.array(frame_with_person['people'][0]['pose_keypoints_2d']).reshape(-1, 2)
    
    plt.figure(figsize=(10, 10))
    plt.scatter(keypoints[:, 0], keypoints[:, 1], c='red', s=100)
    
    # COCO keypoint names (YOLO uses COCO format)
    keypoint_names = [
        'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
        'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
        'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
        'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
    ]
    
    for i, (x, y) in enumerate(keypoints):
        if i < len(keypoint_names):
            plt.annotate(keypoint_names[i], (x, y), fontsize=8)
    
    plt.gca().invert_yaxis()
    plt.title('Detected Keypoints')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.show()

## 6. Build Search Index

In [ ]:
# Initialize search engine
search_engine = PoseSearchEngine(normalize_poses=True)

# Load all videos into search index
for video_path in input_videos:
    video_name = Path(video_path).stem
    json_path = keypoints_dir / f"{video_name}_keypoints.json"
    
    if json_path.exists():
        search_engine.load_video_keypoints(video_name, str(json_path), format='yolo')

# Print database statistics
stats = search_engine.get_statistics()
print("\nSearch Database Statistics:")
print(f"  Videos: {stats['num_videos']}")
print(f"  Total frames: {stats['total_frames']}")
print(f"  Average frames per video: {stats['avg_frames_per_video']:.1f}")
print(f"  Video IDs: {', '.join(stats['video_ids'])}")

## 7. Search for Similar Videos

In [ ]:
# Search for videos similar to 'sample1'
query_video = 'sample1'
results = search_engine.search_similar_videos(query_video, top_k=5, use_dtw=True)

print(f"Videos most similar to '{query_video}':\n")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['video_id']}")
    print(f"   Similarity: {result['similarity']:.3f}")
    print(f"   DTW Distance: {result['distance']:.3f}")
    print(f"   Frames: {result['num_frames']}")
    print()

## 8. Search by Specific Pose

In [ ]:
# Extract a specific pose to search for
with open(keypoints_dir / 'sample1_keypoints.json', 'r') as f:
    data = json.load(f)

# Get a pose from frame 50
target_frame = 50
if data[target_frame]['people']:
    target_pose = np.array(data[target_frame]['people'][0]['pose_keypoints_2d']).reshape(-1, 2)
    
    # Search for similar poses across all videos
    matches = search_engine.search_by_pose(target_pose, threshold=0.7)
    
    print(f"Found {len(matches)} matching poses (similarity >= 0.7):\n")
    for match in matches[:10]:  # Show top 10
        print(f"Video: {match['video_id']}, Frame: {match['frame']}, "
              f"Time: {match['timestamp']:.2f}s, Similarity: {match['similarity']:.3f}")

## 9. Action Segmentation

In [ ]:
# Segment a video into action clips
video_id = 'sample1'
segments = search_engine.segment_by_action(video_id, window_size=30, stride=15)

print(f"Segmented '{video_id}' into {len(segments)} clips:\n")
for i, segment in enumerate(segments[:5]):  # Show first 5
    print(f"Segment {i+1}: Frames {segment['start_frame']}-{segment['end_frame']} "
          f"({segment['duration_frames']} frames)")

## 10. Export Results

In [ ]:
# Export search database index
search_engine.export_database_index(str(output_dir / 'search_index.json'))
print("Search index exported successfully!")

In [ ]:
# Save similarity matrix for all videos
video_ids = stats['video_ids']
similarity_matrix = np.zeros((len(video_ids), len(video_ids)))

for i, vid1 in enumerate(video_ids):
    for j, vid2 in enumerate(video_ids):
        if i == j:
            similarity_matrix[i, j] = 1.0
        elif i < j:
            sim, _ = search_engine.compute_video_similarity_dtw(vid1, vid2)
            similarity_matrix[i, j] = sim
            similarity_matrix[j, i] = sim

# Visualize similarity matrix
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='hot', interpolation='nearest')
plt.colorbar(label='Similarity')
plt.xticks(range(len(video_ids)), video_ids, rotation=45)
plt.yticks(range(len(video_ids)), video_ids)
plt.title('Video Similarity Matrix')
plt.tight_layout()
plt.show()

# Save matrix
np.save(output_dir / 'similarity_matrix.npy', similarity_matrix)

## 11. Summary and Next Steps

You have successfully:
- ✅ Annotated videos with pose estimation
- ✅ Extracted and saved keypoint data
- ✅ Built a searchable video database
- ✅ Searched for similar videos using DTW
- ✅ Found specific poses across videos
- ✅ Segmented videos into action clips

### Next Steps:
1. Process more videos to build a larger database
2. Fine-tune similarity thresholds for your use case
3. Implement real-time pose matching for live video
4. Add action classification on top of pose detection
5. Create a web interface for video search

### Useful Resources:
- Ultralytics Docs: https://docs.ultralytics.com/
- OpenPose: https://github.com/CMU-Perceptual-Computing-Lab/openpose
- DTW Library: https://dtaidistance.readthedocs.io/